In [273]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, Conv2DTranspose, Concatenate, Add, BatchNormalization, Activation
import os
import glob
import tifffile as tiff
# import numpy as np
# import os
# import matplotlib.pyplot as plt
# from sklearn.metrics import precision_score, recall_score, f1_score
from PIL import Image
import rasterio
from sklearn.model_selection import train_test_split;
import re
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
import matplotlib.pyplot as plt
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score


import json

with open('config.json') as json_file:
    config = json.load(json_file)
    corruption = config['corruption_level']


In [274]:
# ---- Reusable building blocks (adapted to support dropout) ----
def conv_block(x, filters, batchnorm=True):
    conv1 = Conv2D(filters, (3, 3), kernel_initializer='he_normal', padding='same')(x)
    if batchnorm:
        conv1 = BatchNormalization(axis=3)(conv1)
    conv1 = Activation('relu')(conv1)

    conv2 = Conv2D(filters, (3, 3), kernel_initializer='he_normal', padding='same')(conv1)
    if batchnorm:
        conv2 = BatchNormalization(axis=3)(conv2)
    conv2 = Activation('relu')(conv2)

    return conv2

def dense_block(inputs, num_filters, dropout_rate=0.0):
    """
    Dense-like block that concatenates input with conv_block output.
    dropout_rate: applied after concatenation (stochastic behavior used for Bayesian approx).
    """
    conv1 = conv_block(inputs, num_filters)
    concat = Concatenate()([inputs, conv1])
    if dropout_rate and dropout_rate > 0.0:
        concat = Dropout(dropout_rate)(concat)  # respects training flag when model called with training=True
    return concat

def residual_conv_block(x, filters, batchnorm=True, dropout_rate=0.0):
    conv1 = Conv2D(filters, (3, 3), kernel_initializer='he_normal', padding='same')(x)
    if batchnorm:
        conv1 = BatchNormalization(axis=3)(conv1)
    conv1 = Activation('relu')(conv1)

    conv2 = Conv2D(filters, (3, 3), kernel_initializer='he_normal', padding='same')(conv1)
    if batchnorm:
        conv2 = BatchNormalization(axis=3)(conv2)
    conv2 = Activation('relu')(conv2)

    # skip/shortcut
    shortcut = Conv2D(filters, kernel_size=(1, 1), kernel_initializer='he_normal', padding='same')(x)
    if batchnorm:
        shortcut = BatchNormalization(axis=3)(shortcut)
    # no activation on shortcut before add (original had Activation on it; kept minimal)
    respath = Add()([shortcut, conv2])

    if dropout_rate and dropout_rate > 0.0:
        respath = Dropout(dropout_rate)(respath)

    respath = Activation('relu')(respath)
    return respath

# ---- Bayesian U-Net builder ----
def bayesian_dense_unet(input_shape, base_filters=64, dropout_rate=0.3):
    """
    Builds a U-Net variant with multiple Dropout layers for MC Dropout (epistemic uncertainty).
    dropout_rate: dropout probability used across the network (applied where appropriate).
    """
    inputs = Input(input_shape)

    # Encoder (dense blocks with dropout)
    conv1 = dense_block(inputs, base_filters, dropout_rate=dropout_rate)
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)

    conv2 = dense_block(pool1, base_filters * 2, dropout_rate=dropout_rate)
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)

    conv3 = dense_block(pool2, base_filters * 4, dropout_rate=dropout_rate)
    pool3 = MaxPooling2D(pool_size=(2, 2))(conv3)

    conv4 = dense_block(pool3, base_filters * 8, dropout_rate=dropout_rate)
    pool4 = MaxPooling2D(pool_size=(2, 2))(conv4)

    # Bottleneck (two convs + dropout)
    conv5 = Conv2D(base_filters * 16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(pool4)
    conv5 = Conv2D(base_filters * 16, (3, 3), kernel_initializer='he_normal', padding='same')(conv5)
    conv5 = Activation('relu')(conv5)
    drop5 = Dropout(dropout_rate)(conv5)  # core stochastic layer

    # Decoder (transpose convs + residual blocks + dropout)
    up6 = Conv2DTranspose(base_filters * 8, (2, 2), strides=(2, 2), padding='same')(drop5)
    up6 = Concatenate()([up6, conv4])
    conv6 = residual_conv_block(up6, base_filters * 8, dropout_rate=dropout_rate)

    up7 = Conv2DTranspose(base_filters * 4, (2, 2), strides=(2, 2), padding='same')(conv6)
    up7 = Concatenate()([up7, conv3])
    conv7 = residual_conv_block(up7, base_filters * 4, dropout_rate=dropout_rate)

    up8 = Conv2DTranspose(base_filters * 2, (2, 2), strides=(2, 2), padding='same')(conv7)
    up8 = Concatenate()([up8, conv2])
    conv8 = residual_conv_block(up8, base_filters * 2, dropout_rate=dropout_rate)

    up9 = Conv2DTranspose(base_filters, (2, 2), strides=(2, 2), padding='same')(conv8)
    up9 = Concatenate()([up9, conv1])
    conv9 = residual_conv_block(up9, base_filters, dropout_rate=dropout_rate)

    # Output - binary segmentation (sigmoid)
    outputs = Conv2D(1, (1, 1), activation='sigmoid')(conv9)

    model = Model(inputs=inputs, outputs=outputs, name='bayesian_dense_unet')
    return model

In [275]:
def dice_coefficient(y_true, y_pred, smooth=1):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    intersection = np.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (np.sum(y_true_f) + np.sum(y_pred_f) + smooth)

def iou(y_true, y_pred, smooth=1):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    intersection = np.sum(y_true_f * y_pred_f)
    union = np.sum(y_true_f) + np.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

def save_image(image, filepath):
    image = (image * 255).astype(np.uint8)  # Scale image to 0-255
    tiff.imwrite(filepath, image)

In [276]:
def predict2_variance(model, base_dir, output_dir, model_type, target_size=(512, 512),
                      number=-1, mc_samples=20, bayesian=False):

    categories = ["Developed", "Natural"]

    dice_scores = []
    iou_scores = []
    precisions = []
    recalls = []
    f1_scores = []

    os.makedirs(output_dir, exist_ok=True)

    TARGET_H, TARGET_W = target_size
    count = 0

    for category in categories:
        image_dir = os.path.join(base_dir, category, "images")
        mask_dir = os.path.join(base_dir, category, "masks")

        mask_files = [f for f in os.listdir(mask_dir) if f.endswith("_mask.tiff")]
        print(f"[{category}] Found {len(mask_files)} mask files")

        for mask_file in mask_files:

            # Extract i from i_mask.tiff
            match = re.match(r"(\d+)_mask\.tiff$", mask_file)
            if not match:
                print(f"Skipping invalid mask filename: {mask_file}")
                continue

            i_str = match.group(1)

            image_file = f"{i_str}.tiff"
            image_path = os.path.join(image_dir, image_file)
            mask_path = os.path.join(mask_dir, mask_file)

            if not os.path.exists(image_path):
                print(f"Missing image for mask {mask_file}")
                continue

            print(f"Processing {category} → {i_str}")

            # ---------- Load Image/Mask ----------
            arbitrary_img = tiff.imread(image_path)
            arbitrary_mask = tiff.imread(mask_path)

            # Expand image to (H, W, 1)
            if len(arbitrary_img.shape) == 2:
                arbitrary_img = np.expand_dims(arbitrary_img, axis=-1)
            elif arbitrary_img.shape[0] == 2:  # if 2-band
                arbitrary_img = arbitrary_img[0]
                arbitrary_img = np.expand_dims(arbitrary_img, axis=-1)

            # ---------- Resize to 512x512 ----------
            # IMAGE → bilinear
            arbitrary_img_resized = cv2.resize(
                arbitrary_img.squeeze(),
                (TARGET_W, TARGET_H),
                interpolation=cv2.INTER_LINEAR
            )
            arbitrary_img_resized = np.expand_dims(arbitrary_img_resized, axis=-1)

            # MASK → nearest neighbor
            arbitrary_mask_resized = cv2.resize(
                arbitrary_mask,
                (TARGET_W, TARGET_H),
                interpolation=cv2.INTER_NEAREST
            )

            # ---------- Bayesian Prediction ----------
            if bayesian:
                preds = []
                for _ in range(mc_samples):
                    pred = model(
                        np.expand_dims(arbitrary_img_resized, axis=0),
                        training=True
                    )[0].numpy()
                    preds.append(pred)

                preds = np.stack(preds, axis=0)
                predicted_mask = np.mean(preds, axis=0)
                uncertainty_map = np.var(preds, axis=0)

                # Save heatmap
                plt.figure(figsize=(5, 5))
                plt.imshow(uncertainty_map.squeeze(), cmap='hot')
                plt.colorbar(label='Uncertainty')
                plt.title(f'Uncertainty Heatmap - {i_str}')
                plt.axis('off')
                plt.savefig(f"{output_dir}/{model_type}_Uncertainty_Heatmap_{i_str}.png",
                            bbox_inches='tight')
                plt.close()

            else:
                predicted_mask = model.predict(
                    np.expand_dims(arbitrary_img_resized, axis=0)
                )[0]

            # ---------- Threshold ----------
            predicted_mask_thresh = (predicted_mask > 0.5).astype(np.uint8)

            # ---------- Save Predictions ----------
            save_image(predicted_mask, f"{output_dir}/{model_type}_Predicted_Image_{i_str}.tif")
            save_image(predicted_mask_thresh, f"{output_dir}/{model_type}_{i_str}.tif")

            # ---------- Visualization ----------
            plt.figure(figsize=(10, 5))

            plt.subplot(2, 3, 1)
            plt.imshow(arbitrary_img_resized.squeeze(), cmap='gray')
            plt.title('Input Image')
            plt.axis('off')

            plt.subplot(2, 3, 2)
            plt.imshow(arbitrary_mask_resized, cmap='gray')
            plt.title('Actual Mask')
            plt.axis('off')

            plt.subplot(2, 3, 3)
            plt.imshow(predicted_mask.squeeze(), cmap='gray')
            plt.title('Predicted Mask (Raw)')
            plt.axis('off')

            plt.subplot(2, 3, 4)
            plt.imshow(predicted_mask_thresh.squeeze(), cmap='gray')
            plt.title('Predicted Mask (Thresh)')
            plt.axis('off')

            if bayesian:
                plt.subplot(2, 3, 5)
                plt.imshow(uncertainty_map.squeeze(), cmap='hot')
                plt.title('Uncertainty Heatmap')
                plt.axis('off')

            plt.tight_layout()
            plt.savefig(f"{output_dir}/{model_type}_Visualization_{i_str}.jpg",
                        bbox_inches='tight')
            plt.close()

            # ---------- Metrics ----------
            dice = dice_coefficient(arbitrary_mask_resized, predicted_mask_thresh)
            iou_score_val = iou(arbitrary_mask_resized, predicted_mask_thresh)
            precision = precision_score(arbitrary_mask_resized.flatten(),
                                        predicted_mask_thresh.flatten())
            recall = recall_score(arbitrary_mask_resized.flatten(),
                                  predicted_mask_thresh.flatten())
            f1 = f1_score(arbitrary_mask_resized.flatten(),
                          predicted_mask_thresh.flatten())

            dice_scores.append(dice)
            iou_scores.append(iou_score_val)
            precisions.append(precision)
            recalls.append(recall)
            f1_scores.append(f1)

            count += 1
            if number != -1 and count >= number:
                break

        if number != -1 and count >= number:
            break

    # ---------- Aggregate Metrics ----------
    mean_dice = np.mean(dice_scores)
    mean_iou = np.mean(iou_scores)
    mean_precision = np.mean(precisions)
    mean_recall = np.mean(recalls)
    mean_f1 = np.mean(f1_scores)

    print(f"Mean Dice Coefficient: {mean_dice:.4f}")
    print(f"Mean IoU: {mean_iou:.4f}")
    print(f"Mean Precision: {mean_precision:.4f}")
    print(f"Mean Recall: {mean_recall:.4f}")
    print(f"Mean F1 Score: {mean_f1:.4f}")

    return mean_dice, mean_iou, mean_precision, mean_recall, mean_f1


In [277]:
def predict2_entropy(model, base_dir, output_dir, model_type, target_size=(512, 512),
                     number=-1, mc_samples=20, bayesian=False,
                     ):

    dice_scores = []
    iou_scores = []
    precisions = []
    recalls = []
    f1_scores = []

    os.makedirs(output_dir, exist_ok=True)

    categories = ["Developed", "Natural"]
    count = 0

    TARGET_H, TARGET_W = target_size

    for category in categories:

        mask_dir = os.path.join(base_dir, category, "masks")
        image_dir = os.path.join(base_dir, category, "images")

        mask_files = [f for f in os.listdir(mask_dir) if f.endswith('_mask.tiff')]
        print(f"[{category}] Found {len(mask_files)} mask files")

        for mask_file in mask_files:

            print(mask_file)

            # Extract index: "12_mask.tiff" → "12"
            i_str = mask_file.replace("_mask.tiff", "")
            image_file = f"{i_str}.tiff"

            mask_path = os.path.join(mask_dir, mask_file)
            image_path = os.path.join(image_dir, image_file)

            # ---------- Load image & mask ----------
            arbitrary_img = tiff.imread(image_path)
            arbitrary_mask = tiff.imread(mask_path)

            # Ensure H × W × 1
            if len(arbitrary_img.shape) == 2:
                arbitrary_img = np.expand_dims(arbitrary_img, axis=-1)
            elif arbitrary_img.shape[0] == 2:    # 2-band TIFF
                arbitrary_img = arbitrary_img[0]
                arbitrary_img = np.expand_dims(arbitrary_img, axis=-1)

            # ---------- Resize (ADDED) ----------
            arbitrary_img_resized = cv2.resize(
                arbitrary_img.squeeze(),
                (TARGET_W, TARGET_H),
                interpolation=cv2.INTER_LINEAR
            )
            arbitrary_img_resized = np.expand_dims(arbitrary_img_resized, axis=-1)

            arbitrary_mask_resized = cv2.resize(
                arbitrary_mask,
                (TARGET_W, TARGET_H),
                interpolation=cv2.INTER_NEAREST
            )

            # ---------- Bayesian Predictive Entropy ----------
            if bayesian:
                preds = []
                for _ in range(mc_samples):
                    p = model(np.expand_dims(arbitrary_img_resized, axis=0),
                              training=True)[0].numpy()
                    preds.append(p)

                preds = np.stack(preds, axis=0)
                predicted_mask = np.mean(preds, axis=0)

                # predictive entropy
                eps = 1e-12
                p = np.clip(predicted_mask, eps, 1 - eps)
                uncertainty_map = -(p * np.log(p) + (1 - p) * np.log(1 - p))

                # Save heatmap
                plt.figure(figsize=(5, 5))
                plt.imshow(uncertainty_map.squeeze(), cmap='hot')
                plt.colorbar(label='Predictive Entropy')
                plt.title(f'Uncertainty Heatmap - {category}_{i_str}')
                plt.axis('off')
                plt.savefig(f"./{output_dir}/{model_type}_Uncertainty_Heatmap_{category}_{i_str}.png",
                            bbox_inches='tight')
                plt.close()

            else:
                predicted_mask = model.predict(
                    np.expand_dims(arbitrary_img_resized, axis=0)
                )[0]

            # ---------- Threshold ----------
            predicted_mask_thresh = (predicted_mask > 0.5).astype(np.uint8)

            # ---------- Save Predictions ----------
            save_image(predicted_mask,
                       f"./{output_dir}/{model_type}_Predicted_Image_{category}_{i_str}.tif")

            save_image(predicted_mask_thresh,
                       f"./{output_dir}/{model_type}_{category}_{i_str}.tif")

            # ---------- Visualization ----------
            plt.figure(figsize=(10, 5))

            plt.subplot(2, 3, 1)
            plt.imshow(arbitrary_img_resized.squeeze(), cmap='gray')
            plt.title('Input Image')
            plt.axis('off')

            plt.subplot(2, 3, 2)
            plt.imshow(arbitrary_mask_resized, cmap='gray')
            plt.title('Actual Mask')
            plt.axis('off')

            plt.subplot(2, 3, 3)
            plt.imshow(predicted_mask.squeeze(), cmap='gray')
            plt.title('Predicted Mask (Raw)')
            plt.axis('off')

            plt.subplot(2, 3, 4)
            plt.imshow(predicted_mask_thresh.squeeze(), cmap='gray')
            plt.title('Predicted Mask (Thresh)')
            plt.axis('off')

            if bayesian:
                plt.subplot(2, 3, 5)
                plt.imshow(uncertainty_map.squeeze(), cmap='hot')
                plt.title('Uncertainty Heatmap')
                plt.axis('off')

            plt.tight_layout()
            plt.savefig(f"./{output_dir}/{model_type}_Visualization_{category}_{i_str}.jpg",
                        bbox_inches='tight')
            plt.close()

            # ---------- Metrics ----------
            dice = dice_coefficient(arbitrary_mask_resized, predicted_mask_thresh)
            iou_score_val = iou(arbitrary_mask_resized, predicted_mask_thresh)
            precision = precision_score(arbitrary_mask_resized.flatten(),
                                        predicted_mask_thresh.flatten())
            recall = recall_score(arbitrary_mask_resized.flatten(),
                                   predicted_mask_thresh.flatten())
            f1 = f1_score(arbitrary_mask_resized.flatten(),
                           predicted_mask_thresh.flatten())

            dice_scores.append(dice)
            iou_scores.append(iou_score_val)
            precisions.append(precision)
            recalls.append(recall)
            f1_scores.append(f1)

            count += 1
            if number != -1 and count >= number:
                break

        if number != -1 and count >= number:
            break

    # ---------- Aggregate Metrics ----------
    print("\n===== METRICS =====")
    print(f"Mean Dice Coefficient: {np.mean(dice_scores):.4f}")
    print(f"Mean IoU: {np.mean(iou_scores):.4f}")
    print(f"Mean Precision: {np.mean(precisions):.4f}")
    print(f"Mean Recall: {np.mean(recalls):.4f}")
    print(f"Mean F1 Score: {np.mean(f1_scores):.4f}")

    return np.mean(dice_scores), np.mean(iou_scores), np.mean(precisions), np.mean(recalls), np.mean(f1_scores)


In [ ]:
def load_data_new(base_dir, target_size=(512, 512), visualize=False):
    categories = ["Developed", "Natural"]
    images = []
    masks = []

    TARGET_H, TARGET_W = target_size

    for category in categories:
        image_dir = os.path.join(base_dir, category, "images")
        mask_dir = os.path.join(base_dir, category, "masks")

        if not os.path.exists(image_dir) or not os.path.exists(mask_dir):
            print(f"Missing directory inside: {category}")
            continue

        # Look for PNG masks instead of TIFF
        mask_files = [f for f in os.listdir(mask_dir) if f.endswith("_mask.png")]

        for mask_file in mask_files:
            # Extract image index (e.g. 123_mask.png → "123")
            match = re.match(r"(\d+)_mask\.png$", mask_file)
            if not match:
                print(f"Skipping invalid mask filename: {mask_file}")
                continue

            i_str = match.group(1)
            image_file = f"{i_str}.png"  # Image is now PNG

            image_path = os.path.join(image_dir, image_file)
            mask_path = os.path.join(mask_dir, mask_file)

            if not os.path.exists(image_path):
                print(f"Missing image for mask {mask_file}")
                continue

            # --------- Read PNG Image & Mask ----------
            img = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
            msk = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)

            if img is None:
                print(f"Error reading {image_path}")
                continue

            if msk is None:
                print(f"Error reading {mask_path}")
                continue

            # Convert mask to single channel if needed
            if len(msk.shape) == 3 and msk.shape[2] > 1:
                msk = cv2.cvtColor(msk, cv2.COLOR_BGR2GRAY)

            # --------- Resize ----------
            img_resized = cv2.resize(img, (TARGET_W, TARGET_H), interpolation=cv2.INTER_LINEAR)
            msk_resized = cv2.resize(msk, (TARGET_W, TARGET_H), interpolation=cv2.INTER_NEAREST)

            # --------- Visualize (optional) ----------
            if visualize:
                plt.figure(figsize=(8, 4))
                plt.subplot(1, 2, 1)
                plt.imshow(img_resized, cmap='gray' if len(img_resized.shape)==2 else None)
                plt.title(f"Image {i_str}")
                plt.axis('off')

                plt.subplot(1, 2, 2)
                plt.imshow(msk_resized, cmap='gray')
                plt.title(f"Mask {i_str}")
                plt.axis('off')
                plt.show()

            # --------- Append ----------
            images.append(img_resized)
            masks.append(msk_resized)

    if not images or not masks:
        print("No images or masks found.")
        return None, None

    images = np.array(images)
    masks = np.array(masks)

    # Normalize mask to 0–1
    masks = masks / 255.0

    print("Loaded images shape:", images.shape)
    print("Loaded masks shape:", masks.shape)

    return images, masks


In [279]:
import matplotlib.pyplot as plt
import numpy as np
import os
from tensorflow.keras.callbacks import Callback

class VisualizePredictions(Callback):
    def __init__(self, val_images, val_masks, output_dir, model_type, num_samples=3):
        super().__init__()
        self.val_images = val_images
        self.val_masks = val_masks
        self.output_dir = output_dir
        self.model_type = model_type
        self.num_samples = num_samples
        os.makedirs(output_dir, exist_ok=True)

    def on_epoch_end(self, epoch, logs=None):
        for i in range(min(self.num_samples, len(self.val_images))):
            img = self.val_images[i]
            mask = self.val_masks[i]

            # Predict
            pred = self.model.predict(np.expand_dims(img, axis=0))[0]
            print(pred)
            pred_thresh = (pred > 0.5).astype(np.uint8)

            # Visualization
            plt.figure(figsize=(10, 4))
            plt.subplot(1, 3, 1)
            plt.imshow(img.squeeze(), cmap='gray')
            plt.title("Input Image")
            plt.axis('off')

            plt.subplot(1, 3, 2)
            plt.imshow(mask.squeeze(), cmap='gray')
            plt.title("Ground Truth")
            plt.axis('off')

            plt.subplot(1, 3, 3)
            plt.imshow(pred_thresh.squeeze(), cmap='gray')
            plt.title("Predicted Mask")
            plt.axis('off')

            plt.tight_layout()
            plt.savefig(os.path.join(
                self.output_dir,
                f"{self.model_type}_epoch{epoch+1}_sample{i}.png"
            ))
            plt.close()

In [280]:
dropout_rate = 0.3
base_dir = f'./Dataset_Coastline/Train/'
target_shape = (512, 512)
# Load data
images, masks = load_data_new(base_dir, target_shape, visualize=False)
target_shape = images.shape[1:]

model_name = f'Bayesian_UNet_Coastline_dense_{dropout_rate}_{corruption}'
print(model_name)

[[[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [1. 1. 1. ... 1. 1. 1.]
  [1. 1. 1. ... 1. 1. 1.]
  [1. 1. 1. ... 1. 1. 1.]]

 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [1. 1. 1. ... 1. 1. 1.]
  [1. 1. 1. ... 1. 1. 1.]
  [1. 1. 1. ... 1. 1. 1.]]

 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [1. 1. 1. ... 1. 1. 1.]
  [1. 1. 1. ... 1. 1. 1.]
  [1. 1. 1. ... 1. 1. 1.]]

 ...

 [[0. 0. 0. ... 1. 1. 1.]
  [0. 0. 0. ... 1. 1. 1.]
  [0. 0. 0. ... 1. 1. 1.]
  ...
  [0. 0. 0. ... 1. 1. 1.]
  [0. 0. 0. ... 1. 1. 1.]
  [0. 0. 0. ... 1. 1. 1.]]

 [[1. 1. 1. ... 0. 0. 0.]
  [1. 1. 1. ... 0. 0. 0.]
  [1. 1. 1. ... 0. 0. 0.]
  ...
  [1. 1. 1. ... 0. 0. 0.]
  [1. 1. 1. ... 0. 0. 0.]
  [1. 1. 1. ... 0. 0. 0.]]

 [[1. 1. 1. ... 1. 1. 1.]
  [1. 1. 1. ... 1. 1. 1.]
  [1. 1. 1. ... 1. 1. 1.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]]
Loaded ima

In [281]:
model = bayesian_dense_unet(target_shape, dropout_rate=dropout_rate)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# early_stopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

# Split data manually to get validation set
X_train, X_val, y_train, y_val = train_test_split(
    images, masks, test_size=0.125, random_state=42
)

visual_cb = VisualizePredictions(
    val_images=X_val,
    val_masks=y_val,
    output_dir="./val_predictions",
    model_type="bayesian_unet",
    num_samples=3
)

model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=2,
    callbacks=[visual_cb]
)

model.save(model_name)

Epoch 1/20
1/1 [==============================] - 0s 304ms/step loss: 0.6881 - accuracy: 0.54


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.005265583..8.547305].


[[[0.5159112]
  [0.5159112]
  [0.5159112]
  ...
  [0.5159112]
  [0.5159112]
  [0.5159112]]

 [[0.5159112]
  [0.5159112]
  [0.5159112]
  ...
  [0.5159112]
  [0.5159112]
  [0.5159112]]

 [[0.5159112]
  [0.5159112]
  [0.5159112]
  ...
  [0.5159112]
  [0.5159112]
  [0.5159112]]

 ...

 [[0.5159112]
  [0.5159112]
  [0.5159112]
  ...
  [0.5159112]
  [0.5159112]
  [0.5159112]]

 [[0.5159112]
  [0.5159112]
  [0.5159112]
  ...
  [0.5159112]
  [0.5159112]
  [0.5159112]]

 [[0.5159112]
  [0.5159112]
  [0.5159112]
  ...
  [0.5159112]
  [0.5159112]
  [0.5159112]]]
1/1 [==============================] - 0s 2ms/step


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.0020450524..17.189941].


[[[0.5159112]
  [0.5159112]
  [0.5159112]
  ...
  [0.5159112]
  [0.5159112]
  [0.5159112]]

 [[0.5159112]
  [0.5159112]
  [0.5159112]
  ...
  [0.5159112]
  [0.5159112]
  [0.5159112]]

 [[0.5159112]
  [0.5159112]
  [0.5159112]
  ...
  [0.5159112]
  [0.5159112]
  [0.5159112]]

 ...

 [[0.5159112]
  [0.5159112]
  [0.5159112]
  ...
  [0.5159112]
  [0.5159112]
  [0.5159112]]

 [[0.5159112]
  [0.5159112]
  [0.5159112]
  ...
  [0.5159112]
  [0.5159112]
  [0.5159112]]

 [[0.5159112]
  [0.5159112]
  [0.5159112]
  ...
  [0.5159112]
  [0.5159112]
  [0.5159112]]]
1/1 [==============================] - 0s 15ms/step


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.005176229..1008.47705].


[[[0.5159112]
  [0.5159112]
  [0.5159112]
  ...
  [0.5159112]
  [0.5159112]
  [0.5159112]]

 [[0.5159112]
  [0.5159112]
  [0.5159112]
  ...
  [0.5159112]
  [0.5159112]
  [0.5159112]]

 [[0.5159112]
  [0.5159112]
  [0.5159112]
  ...
  [0.5159112]
  [0.5159112]
  [0.5159112]]

 ...

 [[0.5159112]
  [0.5159112]
  [0.5159112]
  ...
  [0.5159112]
  [0.5159112]
  [0.5159112]]

 [[0.5159112]
  [0.5159112]
  [0.5159112]
  ...
  [0.5159112]
  [0.5159112]
  [0.5159112]]

 [[0.5159112]
  [0.5159112]
  [0.5159112]
  ...
  [0.5159112]
  [0.5159112]
  [0.5159112]]]
175/175 [==============================] - 78s 436ms/step - loss: 0.6881 - accuracy: 0.5456 - val_loss: 0.6907 - val_accuracy: 0.5464
Epoch 2/20
 36/175 [=====>........................] - ETA: 1:01 - loss: 0.6892 - accuracy: 0.5665

KeyboardInterrupt: 

In [ ]:
base_dir = base_dir = f'./Dataset_Coastline/Test/'
model = bayesian_dense_unet(target_shape, dropout_rate=dropout_rate)
model = tf.keras.models.load_model(model_name, compile=False)
model.compile()

output_dir = f'./GEE_Output/Bayesian_UNet_Coastline/{corruption}/dropout_{dropout_rate}/Unscaled/'
os.makedirs(output_dir,exist_ok=True)

# Run Bayesian inference
predict2_variance(
    model,
    base_dir,
    output_dir,
    f'bayesian_dense',
    mc_samples=20,        # number of MC forward passes
    bayesian=True         # activate Bayesian inference
)

output_dir = f'./GEE_Output/Bayesian_UNet_Coastline/{corruption}/dropout_{dropout_rate}/Unscaled_Entropy/'
os.makedirs(output_dir,exist_ok=True)

predict2_entropy(
    model,
    base_dir,
    output_dir,
    f'bayesian_dense',
    mc_samples=20,        # number of MC forward passes
    bayesian=True         # activate Bayesian inference
)